In [1]:
import io
import zipfile
import pandas as pd
import requests
from IPython.display import display, HTML

In [2]:
urls = []
for year in range(2025, 2027):
    start = 7 if year == 2025 else 1
    end = 13 if year == 2025 else 7
    for i in range(start, end):
        num = f"{i:02d}"
        urls.append(
            f"https://divvy-tripdata.s3.amazonaws.com/{year}{num}-divvy-tripdata.zip"
        )

In [3]:
monthly_dfs = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()
    zip_bytes = io.BytesIO(response.content)
    with zipfile.ZipFile(zip_bytes) as z:
        for filename in z.namelist():
            if filename.endswith(".csv"):
                with z.open(filename) as f:
                    monthly_dfs.append(pd.read_csv(f, encoding="latin-1"))
df = pd.concat(monthly_dfs, ignore_index=True)
print(f"Finish! Total rows concatenated: {len(df)}")

Finish! Total rows concatenated: 5932350


In [4]:
columns_to_drop = df.columns[-3:].tolist()
df_final = df.drop(columns=columns_to_drop)

In [5]:
df_final.to_csv("bike_trip_data(July 2025-June 2026).csv", index=False)

---

In [6]:
df = pd.read_csv("bike_trip_data(July 2025-June 2026).csv")

In [7]:
data_summary = []
data_missing = []
space_str = []
data_coor_outlier = []
ins_archive = []
ins_0 = []

df_sum = pd.DataFrame(data_summary, columns=['Condition','Value'])
df_miss = pd.DataFrame(data_missing, columns=['Variables','Total_Missing'])
df_space = pd.DataFrame(space_str, columns=['Variables', 'Value with Extra Space'])
df_coor_outlier = pd.DataFrame(space_str, columns=['Variables', 'Outlier'])
df_arch = pd.DataFrame(ins_archive, columns=['variables', 'val_w_archive'])
df_0 = pd.DataFrame(ins_0, columns=['variables', 'val_w_0'])
df_dtype = pd.DataFrame(df.dtypes, columns=['type']).reset_index()
df_dtype.rename(columns={'index':'Variables'}, inplace=True)

df_sum.loc[len(df_sum)] = ['Total Records', df.shape[0]]
df_sum.loc[len(df_sum)] = [f'Duplicate Data', df.duplicated().sum()]
df_sum.loc[len(df_sum)] = [f'Need to Swap ("started_at" & "ended_at Value")', (df['started_at'] > df['ended_at']).sum()]

for miss in df.isna():
    df_miss.loc[len(df_miss)] = [miss, df[miss].isna().sum()]

for col in df.columns:
    has_extra_space = df[col].astype(str).str.contains(r'^\s+|\s+$|\s{2,}', regex=True)
    df_problem_rows = df[has_extra_space]  # Rows that contain extra spacing (leading/trailing/double)
    df_space.loc[len(df_space)] = [col, len(df_problem_rows)]

for col in df.columns[8:12]:
    if col[-3:] == 'lat':
        outlier_lat = (round(df[col], 2) <= 41.60) | (round(df[col], 2) >= 42.10)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lat.sum()]
    else:
        outlier_lng = (round(df[col], 2) <= -88.00) | (round(df[col], 2) >= -87.50)
        df_coor_outlier.loc[len(df_coor_outlier)] = [col,outlier_lng.sum()]

for col in df.columns[4:8]:
    archived = df[col].str.contains('Archive', case = False, na=False).sum()
    df_arch.loc[len(df_arch)] = [col,archived]

for col in df.columns[4:8]:
    archived = df[col].str.startswith('0', na=False).sum()
    df_0.loc[len(df_0)] = [col,archived]

In [8]:
# Display the audit results using HTML format
html_str_1 = f"""
<h1> Dataset Check </h1>
<div style="display: grid; gap: 20px;">
    <div style="display: flex; gap: 20px;">
        <div><h3>Summary</h3>{df_sum.to_html()}</div>
        <div><h3>Coordinate Outlier</h3>{df_coor_outlier.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Missing Value</h3>{df_miss.to_html()}</div>
        <div><h3>Extra Spaces</h3>{df_space.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Value with "Archive"</h3>{df_arch.to_html()}</div>
        <div><h3>Value start with "0"</h3>{df_0.to_html()}</div>
    </div>
    <div style="display: flex; gap: 20px;">
        <div><h3>Data Type</h3>{df_dtype.to_html()}</div>
    </div>
</div>
"""

display(HTML(html_str_1))

,Condition,Value
0,Total Records,5932350
1,Duplicate Data,35
2,"Need to Swap (""started_at"" & ""ended_at Value"")",29
,Variables,Outlier
0,start_lat,0
1,start_lng,0
2,end_lat,18
3,end_lng,8
,Variables,Total_Missing
0,ride_id,1


---